## Trabajo de transformaciones de excel, Orange y Python

**Integrantes:** Miguel Vangeas Jose Vanegas  
**Institucion:** Universidad Politecnica Saleciana  
**Fecha:** 25/04/2026

### Introduccion

En esta práctica trabajamos con un conjunto de datos de clientes que incluye información como sexo, edad, país y nivel de satisfacción. La idea principal es preparar estos datos para que puedan usarse mejor en un análisis o en un modelo de aprendizaje automático.

Como parte del proceso, se aplican transformaciones tanto a variables categóricas como numéricas. Para las categóricas se usan técnicas como One-Hot Encoding y codificación ordinal, mientras que para las numéricas se emplean métodos de escalado y normalización.

Con este trabajo se busca entender de forma práctica cómo cambian los datos cuando se preparan correctamente y por qué este paso es tan importante antes de aplicar cualquier modelo.

## 1. Preparacion de datos

### 1.1 Importacion de librerias necesarias

En esta primera parte se cargan las librerias que vamos a usar durante todo el informe. `numpy` y `pandas` nos ayudan a trabajar con los datos, `copy` permite hacer una copia segura del dataset original, y las herramientas de `sklearn` se usan para transformar variables y construir el pipeline de preprocesamiento.

In [1]:

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder, MinMaxScaler
import copy
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
print ('librerias cargadas')

librerias cargadas


### 1.2 Cargar datos y copia de seguridad

Aquí se lee el archivo Excel que contiene la información de los clientes. Después de cargarlo, se revisa cuántas observaciones y variables tiene, y se muestra una pequeña vista previa para confirmar que la lectura salió bien.

Luego se crea una copia del dataset original para trabajar sobre ella sin modificar los datos iniciales, algo que siempre es útil para evitar perder información por error.

In [2]:
archivo_excel = 'Libro1.xlsx'
df_original = pd.read_excel(archivo_excel, sheet_name='Hoja1')
df_original.columns = ['sexo', 'edad', 'pais', 'nivelSatisfaccion']

print('Cantidad de observaciones (clientes):', df_original.shape[0])
print('Cantidad de variables:', df_original.shape[1])
print('Forma del dataset:', df_original.shape)
print(df_original.head())

# Creamos una copia de seguridad para trabajar
df_procesado = copy.deepcopy(df_original)

# 2. SEPARACIÓN DE VARIABLES (Tu lógica de separar Y)
# Guardamos la variable de salida original por si la necesitamos después
Y_original = df_procesado[['nivelSatisfaccion']]

# Creamos una copia de seguridad para trabajar
df_procesado = copy.deepcopy(df_original)
df_procesado['pais'] = df_procesado['pais'].str.strip().str.lower()
df_procesado['nivelSatisfaccion'] = df_procesado['nivelSatisfaccion'].astype(str).str.strip().str.lower()

Cantidad de observaciones (clientes): 6
Cantidad de variables: 4
Forma del dataset: (6, 4)
  sexo  edad     pais nivelSatisfaccion
0    F    65   Brasil          me gusta
1    M    26   España       no me gusta
2    F    21    Chile           neutral
3    M    12  Ecuador       no me gusta
4    F    32   España          me gusta


### 1.3 Definicion de variables estructurales

En este punto se separan las columnas según el tipo de información que contienen. Las variables nominales son las que no tienen un orden natural, las numéricas representan cantidades, y las ordinales sí tienen una secuencia lógica.

Hacer esta clasificación es importante porque cada tipo de variable necesita un tratamiento diferente antes de entrar al modelo.

In [3]:
#  Definición de variables
vars_nominales = ['sexo', 'pais']
vars_numericas = ['edad']
vars_ordinales = ['nivelSatisfaccion']

### 1.4 Funcion analitica de columnas

En esta parte se calcula cuántas columnas tendrá aproximadamente el dataset después de la transformación. La idea es anticipar cómo va a quedar la tabla final una vez que se apliquen las codificaciones a las variables nominales.

Esto ayuda a tener una idea clara de cuántas columnas nuevas se generarán y a comprobar que el proceso de transformación está bien planteado.

In [4]:
def analizar_variables(dataframe, vars_nom, vars_ord):
    total_columnas_originales = len(dataframe.columns)
    total_vars_nominales = len(vars_nom)
    total_nuevas_binarias = 0
    
    for variable in vars_nom:
        cantidad_categorias = dataframe[variable].nunique()
        total_nuevas_binarias += cantidad_categorias
        print(f'Categorías en variable nominal "{variable}": {cantidad_categorias}')
        
    print('Nuevas columnas binarias a crear:', total_nuevas_binarias)
    
    # Cálculo final
    total_columnas_final = total_columnas_originales - total_vars_nominales + total_nuevas_binarias
    return total_columnas_final

# Ejecutamos la función y GUARDAMOS el número
total_columnas_esperadas = analizar_variables(df_procesado, vars_nominales, vars_ordinales)
print('Total de columnas tras la transformación:', total_columnas_esperadas)

Categorías en variable nominal "sexo": 2
Categorías en variable nominal "pais": 4
Nuevas columnas binarias a crear: 6
Total de columnas tras la transformación: 8


### 1.5 Definicion de transformadores

En esta sección se definen las reglas que se aplicarán sobre los datos. Aquí se prepara la forma en la que cada tipo de variable será convertida para que el modelo la pueda interpretar.

Las variables ordinales se transforman respetando su orden, las nominales se convierten en columnas binarias con One-Hot Encoding, y las numéricas se escalan para que queden en un rango más manejable.

In [5]:
# . Definición de Transformadores
# Categóricos
mi_orden_logico = [['no me gusta', 'neutral', 'me gusta']]
trans_ordinal = Pipeline(steps=[('ordinal', OrdinalEncoder(categories=mi_orden_logico))])
trans_nominal = Pipeline(steps=[('one_hot', OneHotEncoder(sparse_output=False, handle_unknown="ignore"))])

preprocesador_categorico = ColumnTransformer(transformers=[
    ('cat_ord', trans_ordinal, vars_ordinales),
    ('cat_nom', trans_nominal, vars_nominales)
], remainder='passthrough', n_jobs=-1)

# Numéricos. Se usa la variable calculada 'total_columnas_esperadas' en el range
trans_minmax = Pipeline(steps=[('minmax', MinMaxScaler(feature_range=(0, 1)))])
preprocesador_minmax = ColumnTransformer(transformers=[
    ('trans_minmax', trans_minmax, list(range(total_columnas_esperadas)))
], remainder='passthrough')

### 1.6 Construccion y ejecucion del pipeline maestro

Aquí se unen todas las transformaciones en una sola secuencia. Primero se aplican las reglas a las variables categóricas y después se escalan los valores numéricos.

- `Pipeline(...):` Encadena los preprocesadores anteriores en un orden estricto. Primero se ejecutan las transformaciones de texto a número (`prep_categorico`), y el resultado numérico pasa inmediatamente al escalado (`prep_escalado`).

- `fit_transform(df_procesado):` Es el motor del script. Aplica todas las reglas definidas sobre el DataFrame de trabajo y ejecuta las transformaciones matemáticas complejas. El resultado es `X_transformado`, una matriz pura de números.

In [6]:
pipe = Pipeline(steps=[
    ('prep_categorico', preprocesador_categorico), 
    ('prep_escalado', preprocesador_minmax)
])

# . Ejecución del Pipeline
X_transformado = pipe.fit_transform(df_procesado)
print('\n********** Pipeline aplicado **********')


********** Pipeline aplicado **********


###  1.7 Reconstrucción del Dataset
Dado que `scikit-learn` devuelve una matriz sin nombres de columnas, este bloque restaura la legibilidad de la tabla.

- Se crea una lista vacía `nombres_columnas_finales` y se van agregando ordenadamente los nombres.

*Primero entran las ordinales.*

- Luego, mediante el método `.get_feature_names_out()`, el script navega dentro del Pipeline y extrae los nombres exactos que el `OneHotEncoder` le asignó a las nuevas columnas binarias.

*Finalmente, se añaden los nombres de las numéricas.*

- `pd.DataFrame(...):` Ensambla la matriz matemática `X_transformado` con la lista de nombres recién recuperada, devolviendo un objeto DataFrame de Pandas completamente estructurado.

In [7]:
# . Reconstrucción de nombres de columnas
nombres_columnas_finales = []

if len(vars_ordinales) != 0:
    nombres_columnas_finales.extend(vars_ordinales)

if len(vars_nominales) != 0:
    #  Se usa 'one_hot' respetando exactamente como se llamó arriba
    nombres_nuevas_vars = pipe.named_steps['prep_categorico'].transformers_[1][1].named_steps['one_hot'].get_feature_names_out(vars_nominales)
    nombres_columnas_finales.extend(nombres_nuevas_vars)

if len(vars_numericas) != 0:
    nombres_columnas_finales.extend(vars_numericas)

print('********** Lista de variables reconstruidas:')
print(nombres_columnas_finales)

# Reconstruimos el DataFrame con Pandas
df_final = pd.DataFrame(data=X_transformado, columns=nombres_columnas_finales)

********** Lista de variables reconstruidas:
['nivelSatisfaccion', 'sexo_F', 'sexo_M', 'pais_brasil', 'pais_chile', 'pais_ecuador', 'pais_españa', 'edad']


### 1.8 Exportación de Resultados
- `.to_excel(...):` Toma el DataFrame final, ya limpio y procesado al 100%, y lo guarda en el disco duro bajo un nuevo nombre (`Dataset_Transformado_.xlsx`). El parámetro `index=False` evita que se guarde la numeración de filas de Pandas como una columna extra.

Por último, el dataset transformado se guarda en un nuevo archivo Excel para no sobrescribir el original. También se muestra una vista previa final para comprobar que todo el proceso se ejecutó correctamente y que los datos quedaron listos para usarse en el siguiente paso del análisis.

In [8]:
# --- 8. IMPLEMENTACIÓN DEL CONCAT (Tu parte solicitada) ---
df_final_con_etiquetas = pd.concat([df_final, Y_original.reset_index(drop=True)], axis=1)

df_final.to_excel('Dataset_Transformado_.xlsx', index=False)
print("\nVista previa de los datos transformados:")
print(df_final.head(6))


Vista previa de los datos transformados:
   nivelSatisfaccion  sexo_F  sexo_M  pais_brasil  pais_chile  pais_ecuador  \
0                1.0     1.0     0.0          1.0         0.0           0.0   
1                0.0     0.0     1.0          0.0         0.0           0.0   
2                0.5     1.0     0.0          0.0         1.0           0.0   
3                0.0     0.0     1.0          0.0         0.0           1.0   
4                1.0     1.0     0.0          0.0         0.0           0.0   
5                0.0     0.0     1.0          0.0         0.0           1.0   

   pais_españa      edad  
0          0.0  1.000000  
1          1.0  0.303571  
2          0.0  0.214286  
3          0.0  0.053571  
4          1.0  0.410714  
5          0.0  0.000000  
